# Kriterion Quant · Value Scanner — Test Connessione EODHD

Notebook per validare il client EODHD in Colab prima di procedere con lo sviluppo.

**Prerequisiti**:
- La tua API key deve essere salvata nei **Colab Secrets** con il nome esatto: `EODHD_API_KEY`
- Icona chiave nella barra laterale sinistra di Colab → Aggiungi nuovo segreto → attiva il toggle "Accesso notebook"

**Cosa fa questo notebook**:
1. Clona il repo `kq-value-scanner` da GitHub
2. Installa le dipendenze
3. Verifica la presenza della API key
4. Esegue 4 test incrementali: EOD prices, Fundamentals, Dividends, Search
5. Stampa report sul consumo di crediti

Costo totale stimato: **~15 crediti** (1 EOD + 10 Fundamentals + 1 Dividends + 1 Search + overhead)

## 1. Setup ambiente

Imposta di seguito lo `GITHUB_REPO_URL` con il tuo repo (se pubblico) oppure salta la cella di clone e carica manualmente i file.

In [ ]:
# Se il repo è già pubblico su GitHub, inserisci l'URL qui sotto.
# Altrimenti commenta queste 2 righe e carica manualmente i file del repo.
GITHUB_REPO_URL = "https://github.com/<TUO_UTENTE>/kq-value-scanner.git"

import os, shutil
if os.path.exists('kq-value-scanner'):
    shutil.rmtree('kq-value-scanner')
!git clone -q $GITHUB_REPO_URL
%cd kq-value-scanner
!ls -la

In [ ]:
# Installa le dipendenze Python
!pip install -q -r requirements.txt
print('\n✓ Dipendenze installate')

## 2. Verifica accesso alla API key

Il client Python cerca la chiave in ordine: parametro esplicito → Colab Secrets → variabile d'ambiente → file `.env`.

In Colab, dovrebbe essere trovata automaticamente nei Secrets.

In [ ]:
from google.colab import userdata

try:
    key = userdata.get('EODHD_API_KEY')
    if key:
        print(f"✓ API key trovata nei Colab Secrets (lunghezza: {len(key)} caratteri)")
        print(f"  Preview: {key[:6]}...{key[-4:]}")
    else:
        print("✗ API key vuota nei Colab Secrets")
except Exception as e:
    print(f"✗ Errore accesso Secrets: {e}")
    print("\nRisolvere: icona chiave nella sidebar → EODHD_API_KEY → toggle 'Accesso notebook' attivo")

## 3. Inizializza il client

Il client legge automaticamente la chiave dai Colab Secrets. Non serve passarla esplicitamente.

In [ ]:
import sys
sys.path.insert(0, '.')

from pipeline.fetch.eodhd_client import EODHDClient

client = EODHDClient()
print('\n✓ Client EODHD inizializzato')

## 4. TEST 1 — EOD Historical Prices

Scarichiamo gli ultimi 30 giorni di AAPL. Atteso: DataFrame con colonne `open`, `high`, `low`, `close`, `adjusted_close`, `volume`.

In [ ]:
import pandas as pd

from_date = (pd.Timestamp.utcnow() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
df = client.get_eod('AAPL.US', from_date=from_date)

print(f'Righe scaricate: {len(df)}')
print(f'Range date: {df.index.min().date()} → {df.index.max().date()}')
print(f'Ultima close: ${df["close"].iloc[-1]:.2f}')
print(f'Volume medio: {df["volume"].mean():,.0f}')
df.tail()

## 5. TEST 2 — Fundamentals

Scarichiamo il blocco fundamentals completo per AAPL. È una risposta JSON di grandi dimensioni (~5-10 MB) con tutte le sezioni: General, Highlights, Valuation, SharesStats, Technicals, Earnings, Financials, ecc.

In [ ]:
fund = client.get_fundamentals('AAPL.US')

print(f'Sezioni top-level ricevute: {len(fund.keys())}')
print(f'Chiavi: {sorted(fund.keys())}\n')

# Estrazione informazioni chiave
gen = fund.get('General', {})
hl = fund.get('Highlights', {})
val = fund.get('Valuation', {})

print(f"Company name      : {gen.get('Name')}")
print(f"Sector            : {gen.get('Sector')}")
print(f"Industry          : {gen.get('Industry')}")
print(f"Exchange          : {gen.get('Exchange')}")
print(f"Market Cap        : {hl.get('MarketCapitalization'):,}" if hl.get('MarketCapitalization') else "MCap N/D")
print(f"PE Ratio          : {hl.get('PERatio')}")
print(f"Forward PE        : {hl.get('ForwardPE')}")
print(f"PEG Ratio         : {hl.get('PEGRatio')}")
print(f"EPS               : {hl.get('EarningsShare')}")
print(f"Profit Margin     : {hl.get('ProfitMargin')}")
print(f"ROE               : {hl.get('ReturnOnEquityTTM')}")
print(f"Revenue TTM       : {hl.get('RevenueTTM'):,}" if hl.get('RevenueTTM') else "Rev N/D")

### 5.1 Esplorazione Financials (Balance Sheet quarterly)

Questa è la sezione critica per i nostri calcoli di Cassa e Debito. Verifichiamo la struttura.

In [ ]:
fin = fund.get('Financials', {})
bs_quarterly = fin.get('Balance_Sheet', {}).get('quarterly', {})

if bs_quarterly:
    dates = sorted(bs_quarterly.keys(), reverse=True)[:4]
    print(f'Ultimi 4 trimestri disponibili: {dates}\n')
    latest = bs_quarterly[dates[0]]
    
    print(f"Snapshot ultimo trimestre ({dates[0]}):")
    print(f"  Cash               : {latest.get('cash'):,}" if latest.get('cash') else "  Cash N/D")
    print(f"  Short-term invest. : {latest.get('shortTermInvestments'):,}" if latest.get('shortTermInvestments') else "  STI N/D")
    print(f"  Long-term debt     : {latest.get('longTermDebt'):,}" if latest.get('longTermDebt') else "  LTD N/D")
    print(f"  Short-term debt    : {latest.get('shortLongTermDebt'):,}" if latest.get('shortLongTermDebt') else "  STD N/D")
    print(f"  Total Assets       : {latest.get('totalAssets'):,}" if latest.get('totalAssets') else "  TA N/D")
    print(f"  Total Liabilities  : {latest.get('totalLiab'):,}" if latest.get('totalLiab') else "  TL N/D")
    print(f"  Stockholders Equity: {latest.get('totalStockholderEquity'):,}" if latest.get('totalStockholderEquity') else "  Eq N/D")
else:
    print('✗ Balance Sheet quarterly non disponibile o vuoto')

## 6. TEST 3 — Dividends history

In [ ]:
div = client.get_dividends('AAPL.US', from_date='2020-01-01')
print(f'Record dividendi: {len(div)}')
if not div.empty:
    print(f'Ultimo dividendo: {div.iloc[-1].to_dict()}')
div.tail()

## 7. TEST 4 — Search ticker

In [ ]:
results = client.search('Apple', limit=5)
for r in results:
    print(f"{r.get('Code'):10s}  {r.get('Name'):40s}  {r.get('Exchange')}")

## 8. Report consumo crediti

Visualizza quanti crediti sono stati consumati in questa sessione di test e il breakdown per endpoint.

In [ ]:
import json

stats = client.get_usage_stats()
print(f"Chiamate totali   : {stats['calls_total']}")
print(f"Crediti stimati   : {stats['credits_total']}")
print(f"Errori            : {stats['errors_total']}")
print('\nBreakdown crediti per endpoint:')
print(json.dumps(stats['credits_by_endpoint'], indent=2))

## 9. Esito

Se tutte le celle sopra hanno eseguito senza errori e i dati sono quelli attesi per AAPL, il client è **validato e operativo**.

Puoi procedere con le sessioni successive di sviluppo:
- `fetch_universe.py` → costruzione universo investibile ~2500 ticker
- `fetch_prices.py` → bulk download prezzi notturno
- `fetch_fundamentals.py` → bulk download fundamentals settimanale

In caso di errori, segnalali nella prossima sessione di sviluppo con Claude.

In [ ]:
client.close()
print('✓ Sessione chiusa correttamente')